# SatQuery: score the trained VLM against the untrained one (GPU, about 1-2 hours)

Scores the LoRA adapter and the untrained Qwen3-VL-2B on the **same** benchmark examples, with the exact
prompts used in training, and writes a side-by-side table (`results/comparison.md`).

**One-time setup**
1. Run `satquery_bench_data.ipynb` once (CPU). 
2. Upload the trained adapter folder (`satquery-vlm-lora/`, with `adapter_config.json` and
   `adapter_model.safetensors`) as a private Kaggle dataset.
3. In this notebook: **Add Input** → the output of `satquery_bench_data` and the adapter dataset.
4. **Accelerator: GPU T4 x2** (or P100), **Internet: On** (downloads the base model), then
   **Save Version → Save & Run All (Commit)**. Download `satquery_vlm_scores.zip` from the Output tab.

In [ ]:
import os, glob, json, random, time, subprocess, zipfile, torch
START = time.time()
SESSION_HOURS = 4.0
SCORE_LIMIT = 400        # examples per model (the benchmark is shuffled, so this mixes all question types)
REPO, OUT, WORK = "/tmp/SatQuery", "/kaggle/working", "/tmp/bench"
assert torch.cuda.is_available(), "No GPU: right panel → Session options → Accelerator → GPU T4 x2"

def minutes_left():
    return (START + SESSION_HOURS * 3600 - time.time()) / 60

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input: add the inputs listed at the top of the notebook"
    return os.path.dirname(hits[0])

BENCH, ADAPTER = find("bench.jsonl"), find("adapter_config.json")
print("benchmark:", BENCH, "\nadapter:  ", ADAPTER)

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Vansh-Vaid/SatQuery.git", REPO], check=True)
os.chdir(REPO)
!pip install -q "transformers>=4.57" peft accelerate zstandard rasterio opencv-python-headless reportlab python-multipart
# Kaggle ships torchao 0.10; current peft refuses any torchao older than 0.16. SatQuery does not use it.
!pip uninstall -y -q torchao
!git log --oneline -1

In [ ]:
!python eval/convert.py bentxt --root {BENCH} --split bench --out {WORK}/bench_eval.jsonl
rows = [json.loads(l) for l in open(f"{WORK}/bench_eval.jsonl")]
random.Random(0).shuffle(rows)          # same order as satquery_vlm.ipynb
with open(f"{WORK}/bench_mix.jsonl", "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in rows)
print(len(rows), "benchmark examples; scoring", min(SCORE_LIMIT, len(rows)), "per model")

def score(name, adapter):
    env = {**os.environ, "SATQUERY_VLM_BACKEND": "hf"}
    env.pop("SATQUERY_VLM_ADAPTER", None)
    if adapter:
        env["SATQUERY_VLM_ADAPTER"] = adapter
    limit = int(max(10, (minutes_left() - 15) / (2 if name == "bench_lora" else 1)))
    print(f"--- {name}: at most {limit} minutes")
    r = subprocess.run(["python", "eval/run_eval.py", "--data", f"{WORK}/bench_mix.jsonl", "--mode", "vlm",
                        "--limit", str(SCORE_LIMIT), "--out", f"{OUT}/results/{name}",
                        "--time-limit-min", str(limit)], env=env)
    if r.returncode != 0:
        print(f"WARNING: scoring {name} failed; its numbers are not valid (see the error above)")

score("bench_lora", ADAPTER)
score("bench_base", None)

In [ ]:
cmp = subprocess.run(["python", "eval/compare_runs.py", f"{OUT}/results/bench_base", f"{OUT}/results/bench_lora",
                      "--names", "base", "LoRA"], capture_output=True, text=True)
print(cmp.stdout or cmp.stderr)
open(f"{OUT}/results/comparison.md", "w").write(cmp.stdout or cmp.stderr)
with zipfile.ZipFile(f"{OUT}/satquery_vlm_scores.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(f"{OUT}/results"):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print(f"Download satquery_vlm_scores.zip from the Output tab. Run time {(time.time() - START) / 60:.0f} min")